In [ ]:
# This line installs the latest version of cvxpy, which is needed to use the
# IPOPT solver below. After the next cvxpy release, just importing cvxpy will be
# sufficient.
!pip install git+https://github.com/cvxpy/cvxpy.git

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# Problem data.
m = 1 # pendulum mass
g = 10 # gravity acceleration
l = 1 # rod length
h = .05 # discretization time step
K = 100 # number of time steps

# Initial and final state.
x0 = np.array([np.pi, 0])
xK = np.array([0, 0])

In [ ]:
# Pendulum dynamics in state-space form.
def f(x, u):
    dq = x[1]
    ddq = (m * g * l * cp.sin(x[0]) + u) / (m * l ** 2)
    return cp.hstack([dq, ddq])

In [ ]:
# Variables.
X = cp.Variable((K, 2)) # matrix with the state at each time step
U = cp.Variable(K - 1) # vector with the contro input at each time step

# Constraints.
constraints = [X[0] == x0, X[-1] == xK] # initial and final conditions
for k in range(K - 1):
    constraints.append(X[k + 1] == X[k] + h * f(X[k], U[k])) # discrete-time dynamics

# Objective function.
obj = h * cp.sum_squares(U)

# Initial guess for the optimization solver.
X.value = np.linspace(x0, xK, K)
U.value = np.zeros(K - 1)

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(nlp=True, solver=cp.IPOPT)

In [ ]:
# Plot state trajectory.
plt.figure()
plt.plot(*X.value.T)
plt.xlabel(r'Angle $q$')
plt.ylabel(r'Angular velocity $\dot q$')
plt.title('Pendulum swing-up trajectory')
plt.grid()

In [ ]:
# Animate pendulum motion.
fig, ax = plt.subplots()
ax.set_xlim(-l - .2, l + .2)
ax.set_ylim(-l - .2, l + .2)
ax.set_aspect('equal')
ax.grid()
line, = ax.plot([], [], 'o-')
def init():
    line.set_data([], [])
    return line,

# Animation update function.
hor = l * np.sin(X.value[:, 0])
vert = l * np.cos(X.value[:, 0])
def update(frame):
    line.set_data([0, hor[frame]], [0, vert[frame]])
    return line,

# Create animation.
ani = FuncAnimation(fig, update, frames=K, init_func=init, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())